In [2]:
#!/usr/bin/env python3

from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import pymc as pm
import arviz as az
import pickle
import traceback

# -----------------------
# Config
# -----------------------
NETID = "k16v981"

BASIN = "gulf_aden"   # arabian_gulf, gulf_oman, red_sea

BASIN_NC = f"/home/{NETID}/my_work/code/arabian_peninsula/bayesian_extremes/data/sst/basin_anoms/era5_sst_anom_{BASIN}_1950_2025.nc"
IDX_CSV  = f"/home/{NETID}/my_work/code/arabian_peninsula/bayesian_extremes/data/sst/roni_dmi_monthly_1950_2025.csv"

OUT_IDATA = f"/home/{NETID}/my_work/code/arabian_peninsula/bayesian_extremes/data/sst/studentt_mean_{BASIN}_roni_dmi_idata.nc"

RANDOM_SEED = 72

ENSO_LAG = 2
IOD_LAG  = 1

STANDARDIZE_Y = False

# -----------------------
# Load basin SST anomalies
# -----------------------
ds = xr.open_dataset(BASIN_NC)
da = ds["sst_anom"]

lat_name = "latitude" if "latitude" in da.coords else "lat"
lon_name = "longitude" if "longitude" in da.coords else "lon"

if float(da[lon_name].max()) > 180:
    lon = da[lon_name]
    lon_new = ((lon + 180) % 360) - 180
    da = da.assign_coords({lon_name: lon_new}).sortby(lon_name)

t0 = pd.DatetimeIndex(pd.to_datetime(da["time"].values))
is_month_start_midnight = (t0.day == 1).all() and (t0.hour == 0).all() and (t0.minute == 0).all()

if not is_month_start_midnight:
    print("⚠️ SST time is not monthly. Resampling to monthly month-start (MS) means...")
    da = da.resample(time="MS").mean(skipna=True)

# JJAS only
jjas_mask = pd.DatetimeIndex(pd.to_datetime(da["time"].values)).month.isin([6, 7, 8, 9])
da = da.isel(time=jjas_mask)

# stack spatial dims
da_st = da.stack(space=(lat_name, lon_name))

# keep wet basin points with at least one finite value
valid_space = np.isfinite(da_st).any("time").values
da_st = da_st.isel(space=valid_space)

Y = da_st.values.astype("float32")   # (T, S)
time = pd.to_datetime(da_st["time"].values)

T, S = Y.shape
print(f"✅ SST loaded: T={T}, S={S}")

space_index = da_st["space"].to_index()
lats = np.array([x[0] for x in space_index], dtype="float32")
lons = np.array([x[1] for x in space_index], dtype="float32")

ds.close()

# -----------------------
# Load ENSO / IOD indices
# -----------------------
idx = pd.read_csv(IDX_CSV)

if "time" in idx.columns:
    idx["time"] = pd.to_datetime(idx["time"])
elif {"year", "month"}.issubset(idx.columns):
    idx["time"] = pd.to_datetime(dict(year=idx["year"], month=idx["month"], day=1))
else:
    raise ValueError(
        f"Index CSV needs either 'time' or ('year','month') columns. Found: {idx.columns.tolist()}"
    )

def pick_col(cols, key):
    cols_l = {c.lower(): c for c in cols}
    for cl, orig in cols_l.items():
        if cl == key or key in cl:
            return orig
    return None

roni_col = pick_col(idx.columns, "roni")
dmi_col  = pick_col(idx.columns, "dmi")
if roni_col is None or dmi_col is None:
    raise ValueError(f"Could not identify RONI/DMI columns. Columns: {idx.columns.tolist()}")

idx = idx.set_index("time").sort_index()

if idx.index.duplicated().any():
    print("⚠️ Duplicate times found in index CSV — deduplicating (keep last).")
    idx = idx[~idx.index.duplicated(keep="last")]

if pd.Index(time).duplicated().any():
    raise ValueError("SST monthly time axis contains duplicates (unexpected).")

# lagged predictors
N_series = idx[roni_col].shift(ENSO_LAG)
D_series = idx[dmi_col].shift(IOD_LAG)

idx_lagged = pd.DataFrame({
    "N_lag": N_series,
    "D_lag": D_series,
}).sort_index()

idx_aligned = idx_lagged.reindex(time)

if idx_aligned[["N_lag", "D_lag"]].isna().any().any():
    missing = idx_aligned[idx_aligned["N_lag"].isna() | idx_aligned["D_lag"].isna()]
    raise ValueError(
        f"Missing lagged index values after aligning to SST months.\n"
        f"First missing rows:\n{missing.head()}\n"
        f"Index range: {idx.index.min()} → {idx.index.max()}\n"
        f"SST range: {time.min()} → {time.max()}\n"
        f"ENSO_LAG={ENSO_LAG}, IOD_LAG={IOD_LAG}"
    )

N = idx_aligned["N_lag"].astype("float32").values
D = idx_aligned["D_lag"].astype("float32").values

# standardize predictors first
N = (N - N.mean()) / N.std()
D = (D - D.mean()) / D.std()

# split ENSO into positive / negative parts
N_pos = np.maximum(N, 0.0).astype("float32")
N_neg = np.minimum(N, 0.0).astype("float32")   # stays negative

# keep IOD linear for now
D = D.astype("float32")

# interactions allowing IOD modulation to differ for El Niño vs La Niña
NposD = (N_pos * D).astype("float32")
NnegD = (N_neg * D).astype("float32")

print(f"✅ Data aligned: T={T} months, S={S} wet points")
print(f"✅ Using lagged predictors: RONI lag={ENSO_LAG}, DMI lag={IOD_LAG}")

# -----------------------
# Optional response standardization
# -----------------------
if STANDARDIZE_Y:
    y_mean = np.nanmean(Y)
    y_sd = np.nanstd(Y)
    Y_model = (Y - y_mean) / y_sd
else:
    y_mean = 0.0
    y_sd = np.nanstd(Y)
    Y_model = Y.copy()

print(f"✅ y_sd for prior scaling = {y_sd:.3f}")

# -----------------------
# Long-form all valid observations
# -----------------------
mask = np.isfinite(Y_model)
t_idx, s_idx = np.where(mask)

y_obs = Y_model[t_idx, s_idx].astype("float32")

Npos_obs = N_pos[t_idx].astype("float32")
Nneg_obs = N_neg[t_idx].astype("float32")
D_obs    = D[t_idx].astype("float32")

NposD_obs = NposD[t_idx].astype("float32")
NnegD_obs = NnegD[t_idx].astype("float32")

s_obs = s_idx.astype("int32")

E = len(y_obs)

print(f"✅ Long table built: n_obs={E}, S={S}")

coords = {
    "obs": np.arange(E),
    "space": np.arange(S),
}

# -----------------------
# Model: hierarchical Student-t mean shift
# -----------------------
# -----------------------
# Model: hierarchical Student-t mean shift
#   ENSO asymmetry allowed via N_pos / N_neg
# -----------------------
with pm.Model(coords=coords) as model:

    y_t     = pm.ConstantData("y_t", y_obs, dims="obs")
    Npos_t  = pm.ConstantData("Npos_t", Npos_obs, dims="obs")
    Nneg_t  = pm.ConstantData("Nneg_t", Nneg_obs, dims="obs")
    D_t     = pm.ConstantData("D_t", D_obs, dims="obs")
    NposD_t = pm.ConstantData("NposD_t", NposD_obs, dims="obs")
    NnegD_t = pm.ConstantData("NnegD_t", NnegD_obs, dims="obs")
    s_id    = pm.ConstantData("s_id", s_obs, dims="obs")

    # prior scales
    mean_scale  = 1.0 if STANDARDIZE_Y else y_sd
    beta_scale  = 0.5 if STANDARDIZE_Y else 0.5 * y_sd
    sigma_scale = 0.5 if STANDARDIZE_Y else 0.5 * y_sd

    # -----------------------
    # Hierarchical intercept
    # -----------------------
    a_bar = pm.Normal("a_bar", mu=0.0, sigma=mean_scale)
    a_sd  = pm.HalfNormal("a_sd", sigma=0.5 * mean_scale)
    a_z   = pm.Normal("a_z", mu=0.0, sigma=1.0, dims="space")
    a_s   = pm.Deterministic("a_s", a_bar + a_sd * a_z, dims="space")

    # -----------------------
    # Hierarchical slopes
    #   N_pos  = El Niño-like side
    #   N_neg  = La Niña-like side
    #   D      = linear IOD effect
    #   NposD  = IOD interaction during El Niño-like states
    #   NnegD  = IOD interaction during La Niña-like states
    # -----------------------
    bNp_bar   = pm.Normal("bNp_bar",   mu=0.0, sigma=beta_scale)
    bNn_bar   = pm.Normal("bNn_bar",   mu=0.0, sigma=beta_scale)
    bD_bar    = pm.Normal("bD_bar",    mu=0.0, sigma=beta_scale)
    bNpD_bar  = pm.Normal("bNpD_bar",  mu=0.0, sigma=beta_scale)
    bNnD_bar  = pm.Normal("bNnD_bar",  mu=0.0, sigma=beta_scale)

    bNp_sd   = pm.HalfNormal("bNp_sd",  sigma=0.5 * beta_scale)
    bNn_sd   = pm.HalfNormal("bNn_sd",  sigma=0.5 * beta_scale)
    bD_sd    = pm.HalfNormal("bD_sd",   sigma=0.5 * beta_scale)
    bNpD_sd  = pm.HalfNormal("bNpD_sd", sigma=0.5 * beta_scale)
    bNnD_sd  = pm.HalfNormal("bNnD_sd", sigma=0.5 * beta_scale)

    bNp_z   = pm.Normal("bNp_z",  mu=0.0, sigma=1.0, dims="space")
    bNn_z   = pm.Normal("bNn_z",  mu=0.0, sigma=1.0, dims="space")
    bD_z    = pm.Normal("bD_z",   mu=0.0, sigma=1.0, dims="space")
    bNpD_z  = pm.Normal("bNpD_z", mu=0.0, sigma=1.0, dims="space")
    bNnD_z  = pm.Normal("bNnD_z", mu=0.0, sigma=1.0, dims="space")

    bNp_s = pm.Deterministic("bNp_s", bNp_bar + bNp_sd * bNp_z, dims="space")
    bNn_s = pm.Deterministic("bNn_s", bNn_bar + bNn_sd * bNn_z, dims="space")
    bD_s  = pm.Deterministic("bD_s",  bD_bar  + bD_sd  * bD_z,  dims="space")

    bNpD_s = pm.Deterministic("bNpD_s", bNpD_bar + bNpD_sd * bNpD_z, dims="space")
    bNnD_s = pm.Deterministic("bNnD_s", bNnD_bar + bNnD_sd * bNnD_z, dims="space")

    # -----------------------
    # Mean model
    # -----------------------
    mu = (
        a_s[s_id]
        + bNp_s[s_id]  * Npos_t
        + bNn_s[s_id]  * Nneg_t
        + bD_s[s_id]   * D_t
        + bNpD_s[s_id] * NposD_t
        + bNnD_s[s_id] * NnegD_t
    )

    # common residual scale
    sigma = pm.HalfNormal("sigma", sigma=sigma_scale)

    # robust df prior: nu > 2
    nu_minus_two = pm.Exponential("nu_minus_two", lam=1 / 10)
    nu = pm.Deterministic("nu", 2.0 + nu_minus_two)

    pm.StudentT(
        "y_like",
        nu=nu,
        mu=mu,
        sigma=sigma,
        observed=y_t,
        dims="obs",
    )

    idata = pm.sample(
        draws=2000,
        tune=2000,
        chains=4,
        cores=4,
        target_accept=0.95,
        random_seed=RANDOM_SEED,
    )

# -----------------------
# Save safely
# -----------------------
def safe_save_idata(idata, out_path):
    try:
        az.to_netcdf(idata, out_path)
        print(f"✅ ArviZ NetCDF saved: {out_path}")
        return
    except Exception as e:
        print("⚠️ az.to_netcdf failed.")
        print(e)
        traceback.print_exc()

    try:
        pkl_path = out_path.replace(".nc", ".pkl")
        with open(pkl_path, "wb") as f:
            pickle.dump(idata, f, protocol=pickle.HIGHEST_PROTOCOL)
        print(f"✅ Pickle saved: {pkl_path}")
    except Exception as e:
        print("❌ Pickle save failed.")
        print(e)

safe_save_idata(idata, OUT_IDATA)

⚠️ SST time is not monthly. Resampling to monthly month-start (MS) means...
✅ SST loaded: T=304, S=710
⚠️ Duplicate times found in index CSV — deduplicating (keep last).
✅ Data aligned: T=304 months, S=710 wet points
✅ Using lagged predictors: RONI lag=2, DMI lag=1
✅ y_sd for prior scaling = 0.582
✅ Long table built: n_obs=215840, S=710


Auto-assigning NUTS sampler...
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [a_bar, a_sd, a_z, bNp_bar, bNn_bar, bD_bar, bNpD_bar, bNnD_bar, bNp_sd, bNn_sd, bD_sd, bNpD_sd, bNnD_sd, bNp_z, bNn_z, bD_z, bNpD_z, bNnD_z, sigma, nu_minus_two]


Sampling 4 chains for 2_000 tune and 2_000 draw iterations (8_000 + 8_000 draws total) took 2648 seconds.


✅ ArviZ NetCDF saved: /home/k16v981/my_work/code/arabian_peninsula/bayesian_extremes/data/sst/studentt_mean_gulf_aden_roni_dmi_idata.nc
